#### https://github.com/HVL-ML/DAT255/blob/main/notebooks/15_gradio_and_streamlit.ipynb er brukt som utgangspunkt i denne notebooken.

# Gradio (and Streamlit) deployment

For deploying an ML-based app there are various approaches, but [Gradio](gradio.app) and [Streamlit](https://streamlit.io/) are both quick and convenient ways to do so. Typically we would implement this in a plain python (`.py`) file rather than a `.ipynb` notebook, but Gradio works here too, so let's try that first. Streamlit needs to be run directly in python, but the code is given below, so you can try out that too.

In this example we set up an image classifier, where the user can upload an image and get the top 5 class predictions in return.

In [350]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras.applications.mobilenet_v2 import (
    MobileNetV2,
    preprocess_input,
    decode_predictions,
)
from PIL import Image

## Set up a pretrained model

Let's download and set up a `MobileNetV2` model, trained on the 1000 classes in the ImageNet dataset. You can change this to anything you like. Remeber, however, to also use the appropriate preprocessing function.

In [ ]:
# model = MobileNetV2(weights="imagenet")

# The model loading below is from Deep Learning with Python, Third edition. Chapter: 8, Fitting the model
model1 = keras.models.load_model(
    "../models/augment.keras"
)
# The model loading below is from Deep Learning with Python, Third edition. Chapter: 8, Fitting the model
model2 = keras.models.load_model(
    "../models/train_hierarchical_cnn_global_aug_v2.keras"
)
# The model loading below is from Deep Learning with Python, Third edition. Chapter: 8, Fitting the model
model3 = keras.models.load_model(
    "../models/hierarchical_cnn_baseline_hybrid_crop.keras"
)

IMG_RES = (300, 300)

# Cellene under kommer tilnærmet direkte fra et annet prosjekt gruppen har arbeidet med i faget, DAT255. [Sett inn referanse]

#### The two cells below are based on Deep Inside Convolutional Networks: Visualising Image Classification Models and Saliency Maps, Image-Specific Class Saliency Visualisation (https://arxiv.org/abs/1312.6034)

In [352]:
# This method is based on Deep Learning with Python, Third edition. Chapter: 2, Gradient computation,
# and get_gradients from https://keras.io/examples/vision/integrated_gradients/
def get_gradients(img_array, lvl, model):
    # img_array /= 255.0
    with tf.GradientTape() as tape:
        tape.watch(img_array)
        # Denne veiledningen på reduce_max ble brukt: https://www.tensorflow.org/api_docs/python/tf/math/reduce_max
        print(img_array)
        score = tf.math.reduce_max(model(img_array)[lvl])
        print(score)
    gradients = tape.gradient(score, img_array)
    return gradients

In [353]:
def get_heatmap(img_array, lvl, model):
    gradients = get_gradients(img_array, lvl, model)
    abs_gradients = np.abs(gradients[0])
    # np.max is from: Deep Learning with Python, Third edition. Chapter: 10, Displaying the class activation heatmap
    max_gradient = np.max(abs_gradients)
    pixel_gradients = ((abs_gradients / max_gradient) * 255.0)

    values = 0
    heatmap = np.zeros(IMG_RES)

    for i in range(len(pixel_gradients)):
        for j in range(len(pixel_gradients[i])):
            # np.max is from: Deep Learning with Python, Third edition. Chapter: 10, Displaying the class activation heatmap
            heatmap[i][j] = np.max(pixel_gradients[i][j])
            values += heatmap[i][j]
    
    return heatmap

### The below cell is strongly based on Deep Learning with Python, Third edition. Chapter: 10, Visualizing heatmaps of class activation

In [354]:
import matplotlib.cm as cm

def superimpose(img_array, heatmap):
    # Uses the "jet" colormap to recolorize the heatmap
    jet = cm.get_cmap("jet")

    jet_colors = jet(np.arange(256))[:, :3]
    jet_colors-=[0,0,0.5]

    # Convertion to int is from: https://www.w3schools.com/python/numpy/numpy_data_types.asp (Converting Data Type on Existing Arrays)
    jet_heatmap = jet_colors[(np.round(heatmap)).astype('i')]

    # Superimposes the heatmap and the original image
    superimposed_img = jet_heatmap + img_array / 3.0
    superimposed_img = keras.utils.array_to_img(superimposed_img)
    return superimposed_img

# Cellene over kommer tilnærmet direkte fra et annet prosjekt gruppen har arbeidet med i faget, DAT255. [Sett inn referanse]

In [ ]:
import imageio
import gradio as gr

from preprocessing import preprocess_image_tensor

def classify_image(img: Image.Image, checkbox_group):

    # Resize to the input image to what MobileNet expects
    # img_resized = img.resize(IMG_RES)
    arr = np.array(img)

    # Check the color channels, so we can take both grayscale, RGB, and RGBA images as input.
    if arr.ndim == 2:
        arr = np.stack([arr] * 3, axis=-1)
    elif arr.shape[-1] == 4:
        arr = arr[..., :3]

    arr = arr / 255.0

    # Linjen under er basert på: https://www.tensorflow.org/api_docs/python/tf/keras/ops/convert_to_tensor
    # arr = keras.ops.convert_to_tensor(arr, dtype="float32")

    arr = preprocess_image_tensor(arr, IMG_RES[0])
    # np.max is from: Deep Learning with Python, Third edition. Chapter: 10, Displaying the class activation heatmap
    arr = arr / np.max(arr)

    # Kodelinjen under kommer delvis fra et annet prosjekt gruppen har arbeidet med i faget, DAT255. [Sett inn referanse]
    arr = keras.ops.expand_dims(arr, 0)

    print(np.max(arr[0]))
    print(np.min(arr[0]))
    # arr = preprocess_input(arr.astype("float32"))

    # Linjen under er delvis basert på: https://www.gradio.app/docs/gradio/checkboxgroup
    return_array = [gr.Image(),gr.Image(),gr.Label(),gr.Label(),gr.Image(),gr.Image(),gr.Label(),gr.Label(),gr.Image(),gr.Image(),gr.Label(),gr.Label(), img, gr.CheckboxGroup()]

    if "Augment.keras (fra John Oscar)" in checkbox_group:
        model = model1
        # Predict!
        preds = model.predict(arr, verbose=0)

        # Kodelinjen under kommer delvis fra et annet prosjekt gruppen har arbeidet med i faget, DAT255. [Sett inn referanse]
        heatmap = get_heatmap(arr, "lvl1", model)
        
        # The below line is based on Deep Learning with Python, Third edition. Chapter: 10, Visualizing heatmaps of class activation
        superimposed = superimpose(arr[0], heatmap)

        # Line below is based on: https://pillow.readthedocs.io/en/stable/reference/Image.html
        superimposed_lvl2 = Image.new(mode="RGB", size=IMG_RES)
        if preds["lvl1"][0][0] > 0.5:
            # Kodelinjen under kommer delvis fra et annet prosjekt gruppen har arbeidet med i faget, DAT255. [Sett inn referanse]
            heatmap_lvl2 = get_heatmap(arr, "lvl2", model)
            # The below line is based on Deep Learning with Python, Third edition. Chapter: 10, Visualizing heatmaps of class activation
            superimposed_lvl2 = superimpose(arr[0], heatmap_lvl2)

        # Line below is based on https://stackoverflow.com/questions/57253048/scipy-misc-has-no-attribute-imsave and https://www.geeksforgeeks.org/python/getting-started-with-imageio-library-in-python/
        # imageio.imwrite("Neinieg.png", superimposed)

        car_model_preds = preds["lvl2"][0]
        return_array[0:4] = [superimposed, superimposed_lvl2, {"Tesla": preds["lvl1"][0][0], "Ikke-Tesla": 1-preds["lvl1"][0][0]}, {'3 2017–2023': car_model_preds[0], '3 2024–nå': car_model_preds[1], 'S 2012–2015': car_model_preds[2], 'S 2016–nå': car_model_preds[3], 'X': car_model_preds[4], 'Y 2020–2024': car_model_preds[5], 'Y 2025-nå': car_model_preds[6]}]

    if "Train_hierarchical_cnn_global_aug_v2.keras (Super god)" in checkbox_group:
        model = model2
        # Predict!
        preds = model.predict(arr, verbose=0)

        # Kodelinjen under kommer delvis fra et annet prosjekt gruppen har arbeidet med i faget, DAT255. [Sett inn referanse]
        heatmap = get_heatmap(arr, "lvl1", model)
        
        # The below line is based on Deep Learning with Python, Third edition. Chapter: 10, Visualizing heatmaps of class activation
        superimposed = superimpose(arr[0], heatmap)

        # Line below is based on: https://pillow.readthedocs.io/en/stable/reference/Image.html
        superimposed_lvl2 = Image.new(mode="RGB", size=IMG_RES)
        if preds["lvl1"][0][0] > 0.5:
            # Kodelinjen under kommer delvis fra et annet prosjekt gruppen har arbeidet med i faget, DAT255. [Sett inn referanse]
            heatmap_lvl2 = get_heatmap(arr, "lvl2", model)
            # The below line is based on Deep Learning with Python, Third edition. Chapter: 10, Visualizing heatmaps of class activation
            superimposed_lvl2 = superimpose(arr[0], heatmap_lvl2)

        # Line below is based on https://stackoverflow.com/questions/57253048/scipy-misc-has-no-attribute-imsave and https://www.geeksforgeeks.org/python/getting-started-with-imageio-library-in-python/
        # imageio.imwrite("Neinieg.png", superimposed)

        car_model_preds = preds["lvl2"][0]
        return_array[4:8] = [superimposed, superimposed_lvl2, {"Tesla": preds["lvl1"][0][0], "Ikke-Tesla": 1-preds["lvl1"][0][0]}, {'3 2017–2023': car_model_preds[0], '3 2024–nå': car_model_preds[1], 'S 2012–2015': car_model_preds[2], 'S 2016–nå': car_model_preds[3], 'X': car_model_preds[4], 'Y 2020–2024': car_model_preds[5], 'Y 2025-nå': car_model_preds[6]}]
    
    if "Horribel modell" in checkbox_group:
        model = model3
        # Predict!
        preds = model.predict(arr, verbose=0)

        # Kodelinjen under kommer delvis fra et annet prosjekt gruppen har arbeidet med i faget, DAT255. [Sett inn referanse]
        heatmap = get_heatmap(arr, "lvl1", model)
        
        # The below line is based on Deep Learning with Python, Third edition. Chapter: 10, Visualizing heatmaps of class activation
        superimposed = superimpose(arr[0], heatmap)

        # Line below is based on: https://pillow.readthedocs.io/en/stable/reference/Image.html
        superimposed_lvl2 = Image.new(mode="RGB", size=IMG_RES)
        if preds["lvl1"][0][0] > 0.5:
            # Kodelinjen under kommer delvis fra et annet prosjekt gruppen har arbeidet med i faget, DAT255. [Sett inn referanse]
            heatmap_lvl2 = get_heatmap(arr, "lvl2", model)
            # The below line is based on Deep Learning with Python, Third edition. Chapter: 10, Visualizing heatmaps of class activation
            superimposed_lvl2 = superimpose(arr[0], heatmap_lvl2)
        # Line below is based on https://stackoverflow.com/questions/57253048/scipy-misc-has-no-attribute-imsave and https://www.geeksforgeeks.org/python/getting-started-with-imageio-library-in-python/
        # imageio.imwrite("Neinieg.png", superimposed)

        car_model_preds = preds["lvl2"][0]
        return_array[8:12] = [superimposed, superimposed_lvl2, {"Tesla": preds["lvl1"][0][0], "Ikke-Tesla": 1-preds["lvl1"][0][0]}, {'3 2017–2023': car_model_preds[0], '3 2024–nå': car_model_preds[1], 'S 2012–2015': car_model_preds[2], 'S 2016–nå': car_model_preds[3], 'X': car_model_preds[4], 'Y 2020–2024': car_model_preds[5], 'Y 2025-nå': car_model_preds[6]}]
    return return_array



#### Cellen under er basert på: https://github.com/gradio-app/gradio/issues/2066 og https://discuss.huggingface.co/t/how-to-programmatically-enable-or-disable-components/52350

In [ ]:
def change_visibility(checkbox_group, model_1_label1, model_1_label2, model_2_label1, model_2_label2, model_3_label1, model_3_label2):
    return_array = [gr.Image(),gr.Image(),gr.Label(),gr.Label(),gr.Markdown(),gr.Image(),gr.Image(),gr.Label(),gr.Label(),gr.Markdown(),gr.Image(),gr.Image(),gr.Label(),gr.Label(),gr.Markdown()]
    if "Augment.keras (fra John Oscar)" in checkbox_group:
        try:
            model_1_label1["Tesla"]
            # The line below is based on: https://www.geeksforgeeks.org/python/python-get-key-with-maximum-value-in-dictionary/
            predicted_model = max(model_1_label2, key=model_1_label2.get)
            return_array[0:5] = [gr.Image(label="Heatmap som viser hvorfor Tesla", visible=True), gr.Image(visible=True, label="Heatmap som viser hvorfor " + predicted_model), gr.Label(visible=True), gr.Label(visible=True), gr.Markdown(visible=True)]
        except:
            return_array[0:5] = [gr.Image(label="Heatmap som viser hvorfor ikke-Tesla", visible=True), gr.Image(visible=False), gr.Label(visible=True), gr.Label(visible=False), gr.Markdown(visible=True)]
    else: 
        return_array[0:5] = [gr.Image(visible=False), gr.Image(visible=False), gr.Label(visible=False), gr.Label(visible=False), gr.Markdown(visible=False)]
    if "Train_hierarchical_cnn_global_aug_v2.keras (Super god)" in checkbox_group:
        try:
            model_2_label1["Tesla"]
            # The line below is based on: https://www.geeksforgeeks.org/python/python-get-key-with-maximum-value-in-dictionary/
            predicted_model = max(model_2_label2, key=model_2_label2.get)
            return_array[5:10] = [gr.Image(label="Heatmap som viser hvorfor Tesla", visible=True), gr.Image(visible=True, label="Heatmap som viser hvorfor " + predicted_model), gr.Label(visible=True), gr.Label(visible=True), gr.Markdown(visible=True)]
        except:
            return_array[5:10] = [gr.Image(label="Heatmap som viser hvorfor ikke-Tesla", visible=True), gr.Image(visible=False), gr.Label(visible=True), gr.Label(visible=False), gr.Markdown(visible=True)]
    else: 
        return_array[5:10] = [gr.Image(visible=False), gr.Image(visible=False), gr.Label(visible=False), gr.Label(visible=False), gr.Markdown(visible=False)]
    if "Horribel modell" in checkbox_group:
        try:
            model_3_label1["Tesla"]
            # The line below is based on: https://www.geeksforgeeks.org/python/python-get-key-with-maximum-value-in-dictionary/
            predicted_model = max(model_3_label2, key=model_3_label2.get)
            return_array[10:15] = [gr.Image(label="Heatmap som viser hvorfor Tesla", visible=True), gr.Image(visible=True, label="Heatmap som viser hvorfor " + predicted_model), gr.Label(visible=True), gr.Label(visible=True), gr.Markdown(visible=True)]
        except:
            return_array[10:15] = [gr.Image(label="Heatmap som viser hvorfor ikke-Tesla", visible=True), gr.Image(visible=False), gr.Label(visible=True), gr.Label(visible=False), gr.Markdown(visible=True)]
    else: 
        return_array[10:15] = [gr.Image(visible=False), gr.Image(visible=False), gr.Label(visible=False), gr.Label(visible=False), gr.Markdown(visible=False)]
    return return_array

## Set up the Gradio interface

Check the [documentation](https://www.gradio.app/docs) for the various things we can add here.

In [ ]:
# Example images
examples = [
    "https://cdn.britannica.com/79/232779-050-6B0411D7/German-Shepherd-dog-Alsatian.jpg",
    "https://cdn.britannica.com/41/126641-050-E1CA0E61/cat-suns-hill-Parthenon-Athens-Greece-Acropolis.jpg",
]

with gr.Blocks(title="Multiattributt visuell kjøretøygjenkjenning under varierende lysforhold") as demo:
    gr.Markdown("## Multiattributt visuell kjøretøygjenkjenning under varierende lysforhold")
    gr.Markdown(
        "Last opp et bilde av et kjøretøy eller velg et eksempel under. "
    )
    with gr.Row():
        components = [[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0],0,0]
        components[3] = gr.Image(type="pil", label="Opplastet bilde")
        # Linjen under er basert på: https://www.gradio.app/docs/gradio/checkboxgroup
        components[4] = gr.CheckboxGroup(choices=["Augment.keras (fra John Oscar)", "Train_hierarchical_cnn_global_aug_v2.keras (Super god)", "Horribel modell"])

    # Linjen under er basert på: https://www.gradio.app/docs/gradio/blocks
    components[0][4] = gr.Markdown("Prediksjonene til Augment.keras (fra John Oscar):", visible=False)
    with gr.Row():
        # De to linjene under er delvis basert på: https://github.com/gradio-app/gradio/issues/10394
        components[0][0] = gr.Image(type="pil", label="Heatmap", format="png", visible=False)
        components[0][1] = gr.Image(type="pil", label="Heatmap", format="png", visible=False)
        # Linjen under er basert på: https://www.gradio.app/4.44.1/guides/controlling-layout
        with gr.Column():
            components[0][2] = gr.Label(num_top_classes=1, label="Bilmerke", visible=False)
            components[0][3] = gr.Label(num_top_classes=3, label="Bilmodell", visible=False)

    # Linjen under er basert på: https://www.gradio.app/docs/gradio/blocks
    components[1][4] = gr.Markdown("Prediksjonene til Train_hierarchical_cnn_global_aug_v2.keras (Super god):", visible=False)
    with gr.Row():
        # De to linjene under er delvis basert på: https://github.com/gradio-app/gradio/issues/10394
        components[1][0] = gr.Image(type="pil", label="Heatmap", format="png", visible=False)
        components[1][1] = gr.Image(type="pil", label="Heatmap", format="png", visible=False)
        # Linjen under er basert på: https://www.gradio.app/4.44.1/guides/controlling-layout
        with gr.Column():
            components[1][2] = gr.Label(num_top_classes=1, label="Bilmerke", visible=False)
            components[1][3] = gr.Label(num_top_classes=3, label="Bilmodell", visible=False)

    # Linjen under er basert på: https://www.gradio.app/docs/gradio/blocks
    components[2][4] = gr.Markdown("Prediksjonene til Horribel modell:", visible=False)
    with gr.Row():
        # De to linjene under er delvis basert på: https://github.com/gradio-app/gradio/issues/10394
        components[2][0] = gr.Image(type="pil", label="Heatmap", format="png", visible=False)
        components[2][1] = gr.Image(type="pil", label="Heatmap", format="png", visible=False)
        # Linjen under er basert på: https://www.gradio.app/4.44.1/guides/controlling-layout
        with gr.Column():
            components[2][2] = gr.Label(num_top_classes=1, label="Bilmerke", visible=False)
            components[2][3] = gr.Label(num_top_classes=3, label="Bilmodell", visible=False)

    classify_btn = gr.Button("Classify", variant="primary")
    classify_btn.click(fn=classify_image, inputs=[components[3],components[4]], outputs=[components[0][0],components[0][1],components[0][2],components[0][3],components[1][0],components[1][1],components[1][2],components[1][3],components[2][0],components[2][1],components[2][2],components[2][3], components[3], components[4]])
    # Linjen under er basert på: https://github.com/gradio-app/gradio/issues/2066 og https://discuss.huggingface.co/t/how-to-programmatically-enable-or-disable-components/52350
    classify_btn.click(fn=change_visibility, inputs=[components[4], components[0][2], components[0][3], components[1][2], components[1][3], components[2][2], components[2][3]], outputs=[components[0][0], components[0][1], components[0][2], components[0][3], components[0][4], components[1][0], components[1][1], components[1][2], components[1][3], components[1][4], components[2][0], components[2][1], components[2][2], components[2][3], components[2][4]])

    examples = gr.Examples(
        examples=examples,
        inputs=components[3],
        # outputs=output,
        # fn=classify_image,
        # cache_examples=True
    )

Now we can run it:

In [ ]:
demo.launch(share=False)

* Running on local URL:  http://127.0.0.1:7910
* To create a public link, set `share=True` in `launch()`.


1.0
1.3770227e-05
tf.Tensor(
[[[[0.88325024 0.93898267 0.8884736 ]
   [0.87888324 0.9403278  0.8952899 ]
   [0.8794407  0.9423987  0.8938633 ]
   ...
   [0.81758183 0.8937849  0.88628083]
   [0.8215509  0.8930346  0.88373965]
   [0.7980084  0.8684293  0.85043305]]

  [[0.88460636 0.9387347  0.888106  ]
   [0.8770344  0.9339554  0.88519716]
   [0.8784819  0.93626004 0.8869326 ]
   ...
   [0.7963991  0.79622245 0.7388848 ]
   [0.8036319  0.80608094 0.7477881 ]
   [0.7650567  0.75989515 0.7061168 ]]

  [[0.8862064  0.93144953 0.8773734 ]
   [0.87045735 0.91627264 0.8563525 ]
   [0.8654833  0.913275   0.85131246]
   ...
   [0.7879578  0.6566903  0.5232604 ]
   [0.804346   0.6681419  0.5260665 ]
   [0.82710063 0.68934876 0.5467919 ]]

  ...

  [[0.4590835  0.4660819  0.47812626]
   [0.42630845 0.42977247 0.43659055]
   [0.48750663 0.4789614  0.47575012]
   ...
   [0.49428546 0.553887   0.6068056 ]
   [0.50854665 0.5626882  0.62310416]
   [0.33914208 0.39158753 0.45654145]]

  [[0.54906064 0

C:\Users\johan\AppData\Local\Temp\ipykernel_4208\2727390636.py:5: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  jet = cm.get_cmap("jet")


tf.Tensor(
[[[[0.88325024 0.93898267 0.8884736 ]
   [0.87888324 0.9403278  0.8952899 ]
   [0.8794407  0.9423987  0.8938633 ]
   ...
   [0.81758183 0.8937849  0.88628083]
   [0.8215509  0.8930346  0.88373965]
   [0.7980084  0.8684293  0.85043305]]

  [[0.88460636 0.9387347  0.888106  ]
   [0.8770344  0.9339554  0.88519716]
   [0.8784819  0.93626004 0.8869326 ]
   ...
   [0.7963991  0.79622245 0.7388848 ]
   [0.8036319  0.80608094 0.7477881 ]
   [0.7650567  0.75989515 0.7061168 ]]

  [[0.8862064  0.93144953 0.8773734 ]
   [0.87045735 0.91627264 0.8563525 ]
   [0.8654833  0.913275   0.85131246]
   ...
   [0.7879578  0.6566903  0.5232604 ]
   [0.804346   0.6681419  0.5260665 ]
   [0.82710063 0.68934876 0.5467919 ]]

  ...

  [[0.4590835  0.4660819  0.47812626]
   [0.42630845 0.42977247 0.43659055]
   [0.48750663 0.4789614  0.47575012]
   ...
   [0.49428546 0.553887   0.6068056 ]
   [0.50854665 0.5626882  0.62310416]
   [0.33914208 0.39158753 0.45654145]]

  [[0.54906064 0.5593098  0.531163

C:\Users\johan\AppData\Local\Temp\ipykernel_4208\2727390636.py:5: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  jet = cm.get_cmap("jet")


tf.Tensor(
[[[[0.3824618  0.41512787 0.4169341 ]
   [0.34720364 0.39493322 0.40596795]
   [0.3754416  0.4312241  0.43561184]
   ...
   [0.1693413  0.23445064 0.32229686]
   [0.16343684 0.21857761 0.2991216 ]
   [0.19212165 0.23606345 0.2950889 ]]

  [[0.56006974 0.58631426 0.5698616 ]
   [0.3931084  0.43718225 0.45994172]
   [0.36881182 0.41884652 0.4367941 ]
   ...
   [0.16803285 0.2362849  0.32632208]
   [0.15685767 0.21687876 0.29710197]
   [0.18409894 0.2344673  0.29850167]]

  [[0.39518738 0.45410666 0.51537997]
   [0.3957848  0.45837253 0.52604747]
   [0.3867755  0.44500917 0.50925523]
   ...
   [0.16668805 0.2367429  0.3263697 ]
   [0.15179008 0.21689877 0.29501462]
   [0.17881845 0.23226579 0.29592752]]

  ...

  [[0.7148454  0.69969034 0.65895075]
   [0.6691631  0.65463054 0.62586164]
   [0.76802903 0.7502874  0.69675475]
   ...
   [0.68378234 0.65899587 0.6209669 ]
   [0.729301   0.7058307  0.6567768 ]
   [0.6719271  0.65445364 0.6176115 ]]

  [[0.6680527  0.65234107 0.620053

C:\Users\johan\AppData\Local\Temp\ipykernel_4208\2727390636.py:5: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  jet = cm.get_cmap("jet")


tf.Tensor(
[[[[0.88325024 0.93898267 0.8884736 ]
   [0.87888324 0.9403278  0.8952899 ]
   [0.8794407  0.9423987  0.8938633 ]
   ...
   [0.81758183 0.8937849  0.88628083]
   [0.8215509  0.8930346  0.88373965]
   [0.7980084  0.8684293  0.85043305]]

  [[0.88460636 0.9387347  0.888106  ]
   [0.8770344  0.9339554  0.88519716]
   [0.8784819  0.93626004 0.8869326 ]
   ...
   [0.7963991  0.79622245 0.7388848 ]
   [0.8036319  0.80608094 0.7477881 ]
   [0.7650567  0.75989515 0.7061168 ]]

  [[0.8862064  0.93144953 0.8773734 ]
   [0.87045735 0.91627264 0.8563525 ]
   [0.8654833  0.913275   0.85131246]
   ...
   [0.7879578  0.6566903  0.5232604 ]
   [0.804346   0.6681419  0.5260665 ]
   [0.82710063 0.68934876 0.5467919 ]]

  ...

  [[0.4590835  0.4660819  0.47812626]
   [0.42630845 0.42977247 0.43659055]
   [0.48750663 0.4789614  0.47575012]
   ...
   [0.49428546 0.553887   0.6068056 ]
   [0.50854665 0.5626882  0.62310416]
   [0.33914208 0.39158753 0.45654145]]

  [[0.54906064 0.5593098  0.531163

C:\Users\johan\AppData\Local\Temp\ipykernel_4208\2727390636.py:5: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  jet = cm.get_cmap("jet")


tf.Tensor(
[[[[0.4497419  0.30944526 0.3091469 ]
   [0.4520526  0.31815937 0.31166244]
   [0.48312226 0.3608638  0.34595832]
   ...
   [0.35582545 0.47234535 0.57433826]
   [0.3602729  0.4588458  0.5507463 ]
   [0.3761458  0.41595712 0.45230344]]

  [[0.4574525  0.31852087 0.31329623]
   [0.46034035 0.32604653 0.31375474]
   [0.53485316 0.4151677  0.38870493]
   ...
   [0.36790273 0.47860497 0.57339346]
   [0.36697602 0.4481976  0.51523566]
   [0.38935852 0.42283696 0.43505636]]

  [[0.45765135 0.31875435 0.3133801 ]
   [0.5016331  0.3668868  0.3548452 ]
   [0.5429861  0.41868374 0.3932356 ]
   ...
   [0.37290284 0.47959906 0.57076114]
   [0.37784013 0.4325295  0.476585  ]
   [0.39448038 0.42438155 0.43430114]]

  ...

  [[0.63459826 0.61446744 0.55623543]
   [0.6204098  0.59525126 0.55155855]
   [0.63259    0.6080369  0.55962825]
   ...
   [0.7029051  0.67064416 0.5943208 ]
   [0.7850304  0.7589931  0.66221935]
   [0.7461238  0.72264504 0.633593  ]]

  [[0.61495864 0.5937738  0.550229